# Robot 1 (R2D2)

In [ ]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 0. Lectura de datos

In [ ]:
DIR = "/content/drive/MyDrive/datasets/properati"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col='id')
df_ent.head(3)

,description,address,lat,lon,publication_date,publisher_id,features,location_0,location_1,location_2,location_3,location_4,operation_type,property_type,source,price,currency_type
id,,,,,,,,,,,,,,,,,
0,galpón de 275 metros cuadrados cubiertos. ...,"Soldini, Rosario, Santa Fe, ARG",-33.008797,-60.743675,"Hace 1 semana, 17 horas",None,electricidad;cerca de:,Argentina,Santa Fe,Soldini,Rosario,None,venta,galpón,properati,150000.0,dolares
1,"venta de galpón en barrio urquiza, rosario sob...","Boulevard 27 de Febrero, Rosario, S2009, Santa...",-32.959099,-60.710079,1 ene 2026,None,1 baño;700 m²;electricidad;cerca de:,Argentina,Santa Fe,Rosario,Rosario,None,venta,galpón,properati,250000.0,dolares
2,mira esta y otras propiedades en nuestro sitio...,"Calle Virasoro 3450, Rosario, S2003, Santa Fe,...",-32.963959,-60.675930,30 nov 2025,None,2 dormitorios;1 baño;250 m²;internet;electrici...,Argentina,Santa Fe,Rosario,Rosario,None,venta,galpón,properati,130000.0,dolares


## 1. Entender los datos (AID)

In [ ]:
df_ent["property_type"].value_counts()

,count
property_type,
departamento,383928
casa,274976
terreno,157493
ph,26985
oficina,22984
local comercial,20375
local,20132
cochera,15444
galpon,10719


In [ ]:
df_ent["location_1"].value_counts().head(20)

,count
location_1,
Buenos Aires,341356
Provincia de Buenos Aires,154995
Capital Federal,138578
Santa Fe,90751
Ciudad Autónoma de Buenos Aires,47534
Córdoba,25626
Departamento de Maldonado,20132
Mendoza,18425
GBA Norte,16125


In [ ]:
df_ent["currency_type"].value_counts()

,count
currency_type,
dolares,910921
Otro,306914
pesos,74839


In [ ]:
df_ent["operation_type"].value_counts()

,count
operation_type,
venta,854477
alquiler,102842
alquiler temporal,20024
temporal,3988
renta,518
alquiler / temporal,328
venta / alquiler,215
venta / temporal,46
sin operacion,25


## 2. Limpiar y transformar los datos (MD)

In [ ]:
df_ent.columns

Index(['description', 'address', 'lat', 'lon', 'publication_date',
       'publisher_id', 'features', 'location_0', 'location_1', 'location_2',
       'location_3', 'location_4', 'operation_type', 'property_type', 'source',
       'price', 'currency_type'],
      dtype='object')

In [ ]:
# selección de datos
#df_ent = df_ent.loc[(df_ent["price"].notna()) & (df_ent["currency_type"] == "dolares") & (df_ent["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA'])) & (df_ent["operation_type"] == 'venta') & (df_ent["property_type"].isin(["departamento", "casa", "departamentos", "casas"]))]
#df_ent.shape

In [ ]:
# selección de datos
tiene_precio = df_ent["price"].notna()
moneda_dolares = df_ent["currency_type"] == 'dolares'
localidades = df_ent["location_1"].isin(["Capital Federal", "Ciudad Autónoma de Buenos Aires", 'CABA'])
tipo_operacion = df_ent["operation_type"] == 'venta'
tipo_propiedad = df_ent["property_type"].isin(["departamento", "casa", "departamentos", "casas"])
#precio_limite_pesos = (df_ent["price"] < 1000000000.0) & (df_ent["currency_type"] == 'pesos')

df_ent = df_ent.loc[tiene_precio & moneda_dolares & localidades & tipo_operacion & tipo_propiedad]
#df_ent = df_ent.loc[tiene_precio & (moneda_dolares | precio_limite_pesos) & localidades & tipo_operacion & tipo_propiedad]
df_ent.shape

(110812, 17)

In [ ]:
# Solo valores numericos
df_ent = df_ent.select_dtypes('number')

# imputación de valores perdidos
#df_ent = df_ent.fillna(0)
df_ent = df_ent.fillna(df_ent.mean())

## 3. Entrenamiento del modelos (AA)

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

## 4. Solución para subir Kaggle

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
#df_ap.head()

In [ ]:
df_ap.shape

(13471, 17)

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

#n_estimators = 500
#max_depth = 50
n_estimators = 50
max_depth = 5

reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo con todos los datos de entrenamiento.csv
reg.fit(X, y)

RandomForestRegressor(max_depth=5, n_estimators=50, n_jobs=-1, random_state=42)

In [ ]:
# Hacemos en df_ap la misma limpieza que en df_ent
#df_ap = df_ap.fillna(0)
df_ap = df_ap.fillna(df_ent.mean())

df_ap = df_ap.select_dtypes('number')

X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap

array([138967.71176938, 134645.9084314 , 135307.41981813, ...,
       181716.2187366 , 134645.9084314 , 170594.45999572])

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
df_ap["price"].to_csv(f"{DIR}/solucion-r2d2.csv")